# §02 Graph Representations — Exercises

> *"The right data structure is worth a thousand algorithms."*

Eight graded exercises covering every representation in the section.
Run cells top-to-bottom; each exercise is self-contained.

**Difficulty:** ★ Foundational · ★★ Intermediate · ★★★ Advanced


In [ ]:
import numpy as np
import json

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [9, 5]
    plt.rcParams['font.size'] = 12
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

try:
    import scipy.sparse as sp
    HAS_SP = True
except ImportError:
    HAS_SP = False

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
print('Setup complete. HAS_MPL =', HAS_MPL, '| HAS_SP =', HAS_SP)


---

## Exercise 1 ★ — Adjacency Matrix Properties

**Difficulty:** ★ Foundational

Consider the undirected graph $G$ with 5 nodes and edges
$E = \{(0,1),(0,2),(1,2),(1,3),(3,4)\}$.

### Part (a)
Build the $5 \times 5$ adjacency matrix $A$ (unweighted, no self-loops).
Verify that $A$ is symmetric and its diagonal is zero.

### Part (b)
Compute $A^2$. What does entry $(A^2)_{ij}$ count?
Verify your answer by counting manually: how many length-2 walks exist from node 1 to node 1?

### Part (c)
Compute the degree vector $\mathbf{d}$ from $A$ using row-sums.
Verify the Handshaking Lemma: $\sum_i d_i = 2|E|$.

### Part (d)
How many non-zero entries does $A$ have? Express this as a fraction of the maximum
possible for a 5-node undirected graph. Is $G$ sparse or dense by this metric?


In [ ]:
# === Exercise 1: Adjacency Matrix Properties ===

n = 5
edges = [(0,1),(0,2),(1,2),(1,3),(3,4)]

# (a) Build adjacency matrix
A = np.zeros((n, n), dtype=int)
for u, v in edges:
    A[u, v] = 1
    A[v, u] = 1

print('A =')
print(A)
sym_ok = np.array_equal(A, A.T)
diag_ok = np.all(np.diag(A) == 0)
print(f'\nSymmetric: {sym_ok} | Zero diagonal: {diag_ok}')

# (b) A^2 — counts length-2 walks
A2 = A @ A
print('\nA^2 =')
print(A2)
print(f'(A^2)[1,1] = {A2[1,1]}  (length-2 walks from node 1 back to node 1)')
# Node 1's neighbours: 0, 2, 3 → three walks 1→x→1
print('Manual count: 1→0→1, 1→2→1, 1→3→1 = 3  ✓')
assert A2[1,1] == 3, f'Expected 3, got {A2[1,1]}'

# (c) Degree vector and Handshaking Lemma
d = A.sum(axis=1)
print(f'\nDegree vector: {d}')
handshake = d.sum() == 2 * len(edges)
print(f'sum(d) = {d.sum()} = 2*{len(edges)} = {2*len(edges)} → Handshaking Lemma: {handshake}')

# (d) Density
nnz = np.count_nonzero(A)
max_edges = n * (n - 1)  # directed; or n*(n-1)/2 undirected — using directed count
density = nnz / (n * (n - 1))
print(f'\nNon-zeros in A: {nnz} out of {n*(n-1)} possible (directed) → density = {density:.3f}')
print('Sparse (density << 1) — sparse storage would save memory for large graphs.')


---

## Exercise 2 ★ — Adjacency List & Edge List Representations

**Difficulty:** ★ Foundational

Use the same graph from Exercise 1: 5 nodes, $E = \{(0,1),(0,2),(1,2),(1,3),(3,4)\}$.

### Part (a)
Build an adjacency list representation using a Python dict of sorted lists.
Print each node's neighbour list. Verify the total number of list entries equals $2|E|$.

### Part (b)
Build an edge list (list of tuples). Extend it to include edge weights
by assigning weight $w(u,v) = u + v + 1$. Print the weighted edge list.

### Part (c)
Convert your weighted edge list to a COO sparse representation:
three arrays `row`, `col`, `data`. Print them and verify their length equals $2|E|$
(one entry per directed half-edge).

### Part (d)
Using only the adjacency list (no adjacency matrix), implement a function
`common_neighbors(adj, u, v)` that returns the set of common neighbours of $u$ and $v$.
Test: `common_neighbors(adj, 0, 1)` should return `{2}`.


In [ ]:
# === Exercise 2: Adjacency List & Edge List ===

edges = [(0,1),(0,2),(1,2),(1,3),(3,4)]
n = 5

# (a) Adjacency list
adj = {i: [] for i in range(n)}
for u, v in edges:
    adj[u].append(v)
    adj[v].append(u)
for node in range(n):
    adj[node].sort()

print('Adjacency list:')
for node, nbrs in adj.items():
    print(f'  {node}: {nbrs}')

total_entries = sum(len(v) for v in adj.values())
print(f'\nTotal list entries: {total_entries} = 2*{len(edges)} = {2*len(edges)} ✓')
assert total_entries == 2 * len(edges)

# (b) Weighted edge list
weighted_edges = [(u, v, u + v + 1) for u, v in edges]
print('\nWeighted edge list (u, v, weight):')
for e in weighted_edges:
    print(f'  {e}')

# (c) COO representation (both directions for undirected)
row, col, data = [], [], []
for u, v, w in weighted_edges:
    row += [u, v]
    col += [v, u]
    data += [w, w]

row = np.array(row)
col = np.array(col)
data = np.array(data)
print(f'\nCOO arrays (len = {len(row)} = 2*{len(edges)}):')
print(f'  row  = {row}')
print(f'  col  = {col}')
print(f'  data = {data}')
assert len(row) == 2 * len(edges), 'COO length mismatch'
print('PASS — COO length verified')

# (d) Common neighbours
def common_neighbors(adj, u, v):
    return set(adj[u]) & set(adj[v])

cn = common_neighbors(adj, 0, 1)
print(f'\ncommon_neighbors(0, 1) = {cn}')
assert cn == {2}, f'Expected {{2}}, got {cn}'
print('PASS — common neighbours verified')


---

## Exercise 3 ★ — Laplacian Matrix & Quadratic Form

**Difficulty:** ★ Foundational

### Part (a)
For the graph from Exercise 1, compute the degree matrix $D = \operatorname{diag}(d_0,\ldots,d_4)$
and the Laplacian $L = D - A$. Print both matrices.

### Part (b)
Verify three Laplacian properties:
1. Every row sums to zero.
2. $L$ is symmetric.
3. $L$ is positive semi-definite (all eigenvalues $\geq 0$).

### Part (c)
Compute the quadratic form $\mathbf{x}^\top L \mathbf{x}$ for $\mathbf{x} = (1, 0, -1, 0, 1)^\top$.
Verify the edge-sum formula: $\mathbf{x}^\top L \mathbf{x} = \sum_{\{i,j\}\in E}(x_i - x_j)^2$.

### Part (d)
The algebraic connectivity (Fiedler value) is $\lambda_2$ — the second-smallest eigenvalue of $L$.
Compute and print $\lambda_1$ and $\lambda_2$. What does $\lambda_2 > 0$ tell you about $G$?


In [ ]:
# === Exercise 3: Laplacian Matrix & Quadratic Form ===

# Rebuild A from Exercise 1
n = 5
edges_list = [(0,1),(0,2),(1,2),(1,3),(3,4)]
A = np.zeros((n, n))
for u, v in edges_list:
    A[u, v] = A[v, u] = 1.0

# (a) Degree matrix and Laplacian
d = A.sum(axis=1)
D = np.diag(d)
L = D - A
print('D =')
print(D.astype(int))
print('\nL = D - A =')
print(L.astype(int))

# (b) Laplacian properties
row_sums = L.sum(axis=1)
sym = np.allclose(L, L.T)
eigvals = np.linalg.eigvalsh(L)
psd = np.all(eigvals >= -1e-10)
print(f'\nRow sums = {row_sums}  (all zero: {np.allclose(row_sums, 0)})')
print(f'Symmetric: {sym}')
print(f'Eigenvalues: {eigvals.round(4)}')
print(f'PSD (all eigenvalues >= 0): {psd}')
assert np.allclose(row_sums, 0), 'Row sums not zero!'
assert sym, 'L not symmetric!'
assert psd, 'L not PSD!'

# (c) Quadratic form
x = np.array([1.0, 0.0, -1.0, 0.0, 1.0])
qf_matrix = x @ L @ x
qf_edges = sum((x[u] - x[v])**2 for u, v in edges_list)
print(f'\nx = {x}')
print(f'x^T L x (matrix)     = {qf_matrix:.4f}')
print(f'x^T L x (edge sum)   = {qf_edges:.4f}')
assert np.isclose(qf_matrix, qf_edges), f'Mismatch: {qf_matrix} vs {qf_edges}'
print('PASS — quadratic form identities match')

# (d) Fiedler value
lambda1 = eigvals[0]    # should be ~0
lambda2 = eigvals[1]    # Fiedler value
print(f'\nlambda_1 = {lambda1:.6f}  (zero eigenvalue = constant eigenvector)')
print(f'lambda_2 = {lambda2:.4f}  (Fiedler value)')
if lambda2 > 1e-10:
    print('lambda_2 > 0 → graph is connected (one connected component)')
else:
    print('lambda_2 = 0 → graph is disconnected')


---

## Exercise 4 ★★ — CSR Format: Manual Construction and Row Access

**Difficulty:** ★★ Intermediate

### Part (a)
For the graph from Exercise 1, manually construct the three CSR arrays:
- `indptr` — row pointer array (length $n+1$)
- `indices` — column indices
- `data` — values (all 1)

Do this from first principles (not using scipy). Print all three arrays.

### Part (b)
Using only the CSR arrays (no matrix), implement a function `csr_row(indptr, indices, data, i)`
that returns the neighbours of node $i$ and their weights as a list of `(col, val)` pairs.
Test: `csr_row(..., 1)` should return neighbours $\{0, 2, 3\}$.

### Part (c)
Implement Sparse Matrix-Vector multiplication $\mathbf{y} = A\mathbf{x}$ using only the CSR arrays
(no `@` operator, no dense matrix). Verify against numpy's matrix multiply.

### Part (d)
Compute the memory in bytes consumed by your CSR representation (int32 arrays) vs the dense
$5 \times 5$ float64 matrix. At what graph size $n$ (assuming $m = 5n$ edges) does CSR break even
with the dense matrix in float64?


In [ ]:
# === Exercise 4: CSR Format — Manual Construction & SpMV ===

n = 5
edges_list = [(0,1),(0,2),(1,2),(1,3),(3,4)]

# Build sorted adjacency list first
adj = {i: [] for i in range(n)}
for u, v in edges_list:
    adj[u].append(v)
    adj[v].append(u)
for i in range(n):
    adj[i].sort()

# (a) Manual CSR construction
indptr = np.zeros(n + 1, dtype=np.int32)
for i in range(n):
    indptr[i + 1] = indptr[i] + len(adj[i])

total_nnz = indptr[-1]
indices = np.zeros(total_nnz, dtype=np.int32)
data_csr = np.ones(total_nnz, dtype=np.float64)

pos = 0
for i in range(n):
    for j in adj[i]:
        indices[pos] = j
        pos += 1

print('CSR arrays:')
print(f'  indptr  = {indptr}  (length {len(indptr)} = n+1)')
print(f'  indices = {indices}  (length {len(indices)} = nnz)')
print(f'  data    = {data_csr}  (all 1)')

# (b) Row access
def csr_row(indptr, indices, data, i):
    start, end = indptr[i], indptr[i+1]
    return [(int(indices[k]), float(data[k])) for k in range(start, end)]

row1 = csr_row(indptr, indices, data_csr, 1)
print(f'\ncsr_row(1) = {row1}')
cols1 = {c for c, _ in row1}
assert cols1 == {0, 2, 3}, f'Expected {{0,2,3}}, got {cols1}'
print('PASS — neighbours of node 1 verified')

# (c) SpMV without @ operator
def csr_spmv(indptr, indices, data, x):
    y = np.zeros(len(indptr) - 1)
    for i in range(len(indptr) - 1):
        start, end = indptr[i], indptr[i+1]
        for k in range(start, end):
            y[i] += data[k] * x[indices[k]]
    return y

A_dense = np.zeros((n, n))
for u, v in edges_list:
    A_dense[u, v] = A_dense[v, u] = 1.0

x_test = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_csr = csr_spmv(indptr, indices, data_csr, x_test)
y_dense = A_dense @ x_test
print(f'\nSpMV result (CSR):   {y_csr}')
print(f'SpMV result (dense): {y_dense}')
assert np.allclose(y_csr, y_dense), 'SpMV mismatch!'
print('PASS — CSR SpMV matches dense multiply')

# (d) Memory comparison
csr_bytes = (n + 1 + total_nnz + total_nnz) * 4  # int32 for indptr+indices, float32 for data
dense_bytes = n * n * 8  # float64
print(f'\nMemory (n=5, nnz={total_nnz}):')
print(f'  CSR (int32 + float32): {csr_bytes} bytes')
print(f'  Dense float64:         {dense_bytes} bytes')
# Break-even: assume m = 5n edges, nnz = 2m = 10n
# CSR_bytes = (n+1+10n+10n)*4 ≈ 84n bytes
# Dense_bytes = 8n^2
# Solve 84n = 8n^2 → n = 84/8 ≈ 10.5
break_even = int(np.ceil(84 / 8))
print(f'  Break-even n (m=5n):   n ≈ {break_even} nodes')
print('  For n > 11 with m=5n edges, CSR saves memory over dense float64.')


---

## Exercise 5 ★★ — Incidence Matrix and $L = BB^\top$

**Difficulty:** ★★ Intermediate

The **incidence matrix** $B \in \{-1,0,1\}^{n \times m}$ of a directed graph assigns
each edge $e_k = (u_k, v_k)$ a column: $B_{u_k,k} = -1$, $B_{v_k,k} = +1$, others 0.

### Part (a)
Orient each edge of the graph from the lower-indexed to the higher-indexed node:
$0 \to 1$, $0 \to 2$, $1 \to 2$, $1 \to 3$, $3 \to 4$.
Build the $5 \times 5$ incidence matrix $B$ and print it.

### Part (b)
Compute $L' = B B^\top$ and verify it equals the undirected Laplacian $L = D - A$
from Exercise 3.

### Part (c)
Show that for any vector $\mathbf{x}$:
$\mathbf{x}^\top B B^\top \mathbf{x} = \|B^\top \mathbf{x}\|^2 = \sum_{\text{edges}} (x_v - x_u)^2$.
Verify numerically with $\mathbf{x} = (2,1,0,3,1)^\top$.

### Part (d)
The null space of $B^\top$ for a connected graph is spanned by $\mathbf{1}$.
Verify: $B^\top \mathbf{1} = \mathbf{0}$ (the all-ones vector maps to zero).


In [ ]:
# === Exercise 5: Incidence Matrix & L = B B^T ===

n = 5
# Directed edges (low → high)
directed_edges = [(0,1),(0,2),(1,2),(1,3),(3,4)]
m = len(directed_edges)

# (a) Incidence matrix
B = np.zeros((n, m), dtype=float)
for k, (u, v) in enumerate(directed_edges):
    B[u, k] = -1.0   # tail (source)
    B[v, k] = +1.0   # head (target)

print('Incidence matrix B (n x m):')
print(B.astype(int))

# (b) L' = B B^T
L_incidence = B @ B.T

# Rebuild L = D - A
A = np.zeros((n, n))
for u, v in directed_edges:
    A[u, v] = A[v, u] = 1.0
D = np.diag(A.sum(axis=1))
L_DA = D - A

print('\nL = B B^T =')
print(L_incidence.astype(int))
print('\nL = D - A =')
print(L_DA.astype(int))
match = np.allclose(L_incidence, L_DA)
print(f'\nL = BB^T matches L = D-A: {match}')
assert match, 'Incidence-based L does not match D-A!'
print('PASS')

# (c) Quadratic form via incidence
x = np.array([2.0, 1.0, 0.0, 3.0, 1.0])
Btx = B.T @ x  # edge potential differences
qf_via_norm = float(np.dot(Btx, Btx))
qf_matrix = float(x @ L_incidence @ x)
qf_edges = sum((x[v] - x[u])**2 for u, v in directed_edges)
print(f'\nx = {x}')
print(f'B^T x (edge differences) = {Btx}')
print(f'||B^T x||^2  = {qf_via_norm:.4f}')
print(f'x^T L x      = {qf_matrix:.4f}')
print(f'sum (x_v-x_u)^2 = {qf_edges:.4f}')
assert np.isclose(qf_via_norm, qf_matrix) and np.isclose(qf_matrix, qf_edges)
print('PASS — all three forms agree')

# (d) Null space of B^T
ones = np.ones(n)
Bt_ones = B.T @ ones
print(f'\nB^T 1 = {Bt_ones}  (should be zero vector)')
assert np.allclose(Bt_ones, 0), 'B^T @ 1 is not zero!'
print('PASS — B^T 1 = 0 confirmed: 1 is in null space of B^T')


---

## Exercise 6 ★★ — Normalised Adjacency & GCN Layer

**Difficulty:** ★★ Intermediate

The normalised adjacency matrix is $\hat{A} = D^{-1/2} A D^{-1/2}$.
The GCN propagation rule (Kipf & Welling 2017) uses
$\tilde{A} = D^{-1/2}(A + I)D^{-1/2}$ (with self-loops).

### Part (a)
Compute $\hat{A}$ for the graph from Exercise 1. Print it and verify:
- All diagonal entries of $\hat{A}$ are 0 (no self-loops).
- All eigenvalues lie in $[-1, 1]$.

### Part (b)
Compute $\tilde{A}$ (GCN version with self-loops). Print and verify eigenvalues lie in $(0, 1]$.

### Part (c)
Simulate one GCN layer: $H^{(1)} = \sigma(\tilde{A} H^{(0)} W)$
where $H^{(0)} \in \mathbb{R}^{5 \times 2}$ is a random feature matrix,
$W \in \mathbb{R}^{2 \times 3}$ is a random weight matrix, and $\sigma = \text{ReLU}$.
Print the output shape and confirm it is $(5, 3)$.

### Part (d)
For a star graph $K_{1,4}$ (one hub, four leaves), compute $\hat{A}$.
What is the entry $\hat{A}_{\text{hub},\text{leaf}}$? Explain intuitively
why degree normalisation "dilutes" the hub's signal in GNN aggregation.


In [ ]:
# === Exercise 6: Normalised Adjacency & GCN Layer ===

n = 5
edges_list = [(0,1),(0,2),(1,2),(1,3),(3,4)]
A = np.zeros((n, n))
for u, v in edges_list:
    A[u, v] = A[v, u] = 1.0

d = A.sum(axis=1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(d))

# (a) Normalised adjacency A_hat
A_hat = D_inv_sqrt @ A @ D_inv_sqrt
print('Normalised adjacency A_hat:')
print(A_hat.round(4))

diag_zero = np.allclose(np.diag(A_hat), 0)
eig_Ahat = np.linalg.eigvalsh(A_hat)
eig_in_range = np.all(eig_Ahat >= -1 - 1e-10) and np.all(eig_Ahat <= 1 + 1e-10)
print(f'Diagonal is zero: {diag_zero}')
print(f'Eigenvalues of A_hat: {eig_Ahat.round(4)}')
print(f'All in [-1,1]: {eig_in_range}')

# (b) GCN A_tilde (with self-loops)
A_tilde_raw = A + np.eye(n)
d_tilde = A_tilde_raw.sum(axis=1)
D_tilde_inv_sqrt = np.diag(1.0 / np.sqrt(d_tilde))
A_tilde = D_tilde_inv_sqrt @ A_tilde_raw @ D_tilde_inv_sqrt
print('\nGCN A_tilde (with self-loops):')
print(A_tilde.round(4))

eig_Atilde = np.linalg.eigvalsh(A_tilde)
print(f'Eigenvalues of A_tilde: {eig_Atilde.round(4)}')
print(f'All in (0,1]: {np.all(eig_Atilde > -1e-10) and np.all(eig_Atilde <= 1+1e-10)}')

# (c) One GCN layer
np.random.seed(7)
H0 = np.random.randn(n, 2)   # 5 nodes, 2 input features
W  = np.random.randn(2, 3)   # 2 → 3 features

def relu(x):
    return np.maximum(0, x)

H1 = relu(A_tilde @ H0 @ W)
print(f'\nH^(1) shape: {H1.shape}  (expected (5, 3))')
assert H1.shape == (5, 3), f'Shape error: {H1.shape}'
print('H^(1) =')
print(H1.round(4))
print('PASS — GCN layer output shape confirmed')

# (d) Star graph K_{1,4}
n_star = 5
A_star = np.zeros((n_star, n_star))
for leaf in range(1, n_star):
    A_star[0, leaf] = A_star[leaf, 0] = 1.0

d_star = A_star.sum(axis=1)
D_star_inv_sqrt = np.diag(1.0 / np.sqrt(d_star))
A_hat_star = D_star_inv_sqrt @ A_star @ D_star_inv_sqrt

print(f'\nStar K_{{1,4}} normalised adjacency A_hat:')
print(A_hat_star.round(4))
hub_leaf = A_hat_star[0, 1]
print(f'A_hat[hub, leaf] = {hub_leaf:.4f}')
print(f'  = 1 / sqrt(d_hub * d_leaf) = 1/sqrt({int(d_star[0])}*{int(d_star[1])}) = {1/np.sqrt(d_star[0]*d_star[1]):.4f}')
print('Intuition: hub has degree 4 → its signal is divided by sqrt(4)=2, diluting each edge.')


---

## Exercise 7 ★★★ — Sparse Format Benchmark & Memory Analysis

**Difficulty:** ★★★ Advanced

In this exercise you will empirically compare dense vs sparse storage and measure
SpMV performance on random sparse graphs.

### Part (a)
Generate Erdős–Rényi random graphs $G(n, p)$ for $n \in \{100, 500, 1000, 2000\}$
with $p = 5/n$ (expected degree 5). For each, compute:
- Dense matrix memory (float64)
- CSR matrix memory (using scipy sparse, int32 + float64)
- Density (fraction of non-zero entries)

### Part (b)
Implement a pure Python CSR SpMV and measure its runtime vs numpy's dense `@` operator
for $n = 500$. Report the speedup factor.

### Part (c)
For a graph with $n = 1000$ and $m = 5000$ edges, analytically compute the crossover
density $d^* = m/(n^2)$ below which CSR uses less memory than dense float64.
Verify your formula: CSR bytes $\approx 8 \cdot 2m + 4(n+1)$ for float64 values.

### Part (d)
(Visualisation) Plot memory usage (MB) vs $n$ for both dense and CSR on a log-log scale.
Add a vertical line at $n = 10{,}000$. What is the approximate memory ratio at that point?


In [ ]:
# === Exercise 7: Sparse Format Benchmark & Memory Analysis ===

import time

# (a) Memory comparison across graph sizes
sizes = [100, 500, 1000, 2000]
results = []

for n in sizes:
    p = 5.0 / n
    rng = np.random.default_rng(42)
    # Generate upper-triangle edges for undirected graph
    mask = rng.random((n, n)) < p
    mask = np.triu(mask, 1)
    mask = mask | mask.T  # symmetrise
    np.fill_diagonal(mask, 0)

    nnz = int(mask.sum())
    density = nnz / (n * n)

    dense_bytes = n * n * 8  # float64
    # CSR: indptr (int32, n+1), indices (int32, nnz), data (float64, nnz)
    csr_bytes = (n + 1) * 4 + nnz * 4 + nnz * 8

    results.append({
        'n': n, 'nnz': nnz, 'density': density,
        'dense_MB': dense_bytes / 1e6,
        'csr_MB': csr_bytes / 1e6,
        'ratio': dense_bytes / csr_bytes
    })

print(f'{'n':>6} {'nnz':>8} {'density':>10} {'dense MB':>10} {'CSR MB':>8} {'ratio':>7}')
print('-' * 60)
for r in results:
    print(f"{r['n']:>6} {r['nnz']:>8} {r['density']:>10.5f} "
          f"{r['dense_MB']:>10.3f} {r['csr_MB']:>8.4f} {r['ratio']:>7.1f}x")


In [ ]:
# (b) SpMV timing: pure Python CSR vs numpy dense

n = 500
p = 5.0 / n
rng = np.random.default_rng(0)
mask = rng.random((n, n)) < p
mask = np.triu(mask, 1); mask = mask | mask.T
np.fill_diagonal(mask, 0)
A_dense_bench = mask.astype(np.float64)

# Build CSR manually
adj_bench = {i: list(np.where(mask[i])[0]) for i in range(n)}
indptr_bench = np.zeros(n + 1, dtype=np.int32)
for i in range(n):
    indptr_bench[i+1] = indptr_bench[i] + len(adj_bench[i])
indices_bench = np.concatenate([adj_bench[i] for i in range(n)]).astype(np.int32)
data_bench = np.ones(len(indices_bench))

x_bench = rng.random(n)

# Dense timing (10 reps)
reps = 10
t0 = time.perf_counter()
for _ in range(reps):
    y_dense = A_dense_bench @ x_bench
t_dense = (time.perf_counter() - t0) / reps

# scipy CSR timing (if available)
if HAS_SP:
    A_sp = sp.csr_matrix(A_dense_bench)
    t0 = time.perf_counter()
    for _ in range(reps):
        y_sp = A_sp @ x_bench
    t_sp = (time.perf_counter() - t0) / reps
    print(f'Dense SpMV:   {t_dense*1e3:.4f} ms')
    print(f'SciPy CSR:    {t_sp*1e3:.4f} ms')
    print(f'Speedup (CSR over dense): {t_dense/t_sp:.2f}x')
    ok = np.allclose(y_dense, y_sp)
    print(f'Results match: {ok}')
else:
    print('scipy not available — showing dense timing only')
    print(f'Dense SpMV: {t_dense*1e3:.4f} ms')


In [ ]:
# (c) Crossover density formula

# CSR bytes = 4*(n+1) + 4*nnz + 8*nnz = 4*(n+1) + 12*nnz
# Dense bytes = 8*n^2
# Crossover: 4*(n+1) + 12*nnz = 8*n^2
# For large n: 12*nnz ≈ 8*n^2 → nnz = 2*n^2/3
# density d* = nnz/n^2 = 2/3 ≈ 0.667

n_test = 1000
m_test = 5000
density_test = m_test / (n_test**2)

dense_bytes = n_test**2 * 8
# nnz = 2*m (both directions for undirected)
nnz_test = 2 * m_test
csr_bytes_test = 4*(n_test+1) + 4*nnz_test + 8*nnz_test

print(f'n = {n_test}, m = {m_test}')
print(f'density = {density_test:.5f}')
print(f'Dense float64: {dense_bytes/1e6:.2f} MB')
print(f'CSR (int32 idx + float64 val): {csr_bytes_test/1e6:.3f} MB')
print(f'CSR is {dense_bytes/csr_bytes_test:.0f}x smaller at this density')

# Crossover formula
# 12*nnz = 8*n^2 → d* = nnz/n^2 = 8/12 = 2/3
d_star = 2.0 / 3.0
print(f'\nCrossover density d* = 2/3 ≈ {d_star:.3f}')
print('Below d* → CSR saves memory; above d* → dense is equally efficient.')
print(f'At density {density_test:.5f} (well below {d_star:.3f}), CSR wins decisively.')


In [ ]:
# (d) Log-log memory plot

if HAS_MPL:
    ns = np.logspace(1, 5, 50).astype(int)
    p_fixed = 5.0 / ns  # expected degree 5
    nnz_arr = (ns * 5).astype(int) * 2  # 2*m ≈ 2*5n

    dense_MB = ns**2 * 8 / 1e6
    csr_MB = (4*(ns+1) + 4*nnz_arr + 8*nnz_arr) / 1e6

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.loglog(ns, dense_MB, color='#CC3311', lw=2.5, label='Dense float64')
    ax.loglog(ns, csr_MB, color='#0077BB', lw=2.5, label='CSR (int32+float64, m=5n)')
    ax.axvline(x=10000, color='#555555', ls='--', alpha=0.7, label='n = 10,000')

    idx = np.searchsorted(ns, 10000)
    ratio_at_10k = dense_MB[idx] / csr_MB[idx]
    ax.annotate(f'≈ {ratio_at_10k:.0f}× smaller',
                xy=(10000, csr_MB[idx]), xytext=(3000, csr_MB[idx]*200),
                arrowprops=dict(arrowstyle='->', color='#555555'),
                fontsize=11, color='#0077BB')

    ax.set_xlabel('Number of nodes n', fontsize=12)
    ax.set_ylabel('Memory (MB)', fontsize=12)
    ax.set_title('Memory Usage: Dense vs CSR (m = 5n edges)', fontsize=13)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
    print(f'At n=10,000: dense uses {dense_MB[idx]:.0f} MB vs CSR {csr_MB[idx]:.2f} MB (≈{ratio_at_10k:.0f}× ratio)')
else:
    print('Matplotlib not available — skip plot')
    ns = np.array([100, 1000, 10000])
    nnz_arr = ns * 10
    for n_val, nnz_val in zip(ns, nnz_arr):
        dense = n_val**2 * 8 / 1e6
        csr = (4*(n_val+1) + 12*nnz_val) / 1e6
        print(f'n={n_val:>6}: dense={dense:>8.2f} MB, CSR={csr:>6.3f} MB, ratio={dense/csr:>6.1f}x')


---

## Exercise 8 ★★★ — Heterogeneous Graph & Message Passing

**Difficulty:** ★★★ Advanced

Heterogeneous graphs have multiple node types and edge relation types.
This exercise builds a mini knowledge graph and implements relation-specific
message passing (RGCN-style).

### Part (a)
Build a small academic graph with:
- **3 Author** nodes: IDs 0, 1, 2
- **2 Paper** nodes: IDs 0, 1
- **2 Venue** nodes: IDs 0, 1
- **Relation `writes`**: Author 0 → Paper 0; Author 1 → Paper 0; Author 2 → Paper 1
- **Relation `published_at`**: Paper 0 → Venue 0; Paper 1 → Venue 1
- **Relation `co-author`**: Author 0 ↔ Author 1 (undirected)

Represent this as a dict of COO edge lists: `edges['writes'] = [(src_type, src_id, dst_type, dst_id), ...]`.

### Part (b)
Build per-relation adjacency matrices:
- $A_{\text{writes}} \in \{0,1\}^{3 \times 2}$ (Author → Paper)
- $A_{\text{pub}} \in \{0,1\}^{2 \times 2}$ (Paper → Venue)
- $A_{\text{coauth}} \in \{0,1\}^{3 \times 3}$ (Author ↔ Author)

### Part (c)
Implement one step of **RGCN aggregation** for paper nodes:
$$h_{\text{paper}}^{(1)} = \sigma\!\left(W_{\text{self}} H_{\text{paper}} + A_{\text{writes}}^\top H_{\text{author}} W_{\text{writes}}\right)$$
Use random $2$-dimensional features for all nodes and random $2 \times 2$ weight matrices.
Print the resulting $2 \times 2$ paper embedding matrix.

### Part (d)
Count the total number of parameters in an RGCN with:
- 3 relations, each with its own $d_{\text{in}} \times d_{\text{out}}$ weight matrix ($d=2$)
- 1 self-loop weight per node type (3 types)
- Bias terms for each node type

Compare to a homogeneous GCN with one shared weight matrix. What is the parameter overhead?


In [ ]:
# === Exercise 8: Heterogeneous Graph & RGCN Message Passing ===

# (a) Heterogeneous graph as COO edge lists
edges = {
    'writes': [(0, 0), (1, 0), (2, 1)],          # (author_id, paper_id)
    'published_at': [(0, 0), (1, 1)],             # (paper_id, venue_id)
    'co_author': [(0, 1), (1, 0)],               # (author_id, author_id) — undirected
}

print('Heterogeneous graph edge lists:')
for rel, elist in edges.items():
    print(f'  [{rel}]: {elist}')

# (b) Per-relation adjacency matrices
n_author, n_paper, n_venue = 3, 2, 2

A_writes = np.zeros((n_author, n_paper))
for a, p in edges['writes']:
    A_writes[a, p] = 1.0

A_pub = np.zeros((n_paper, n_venue))
for p, v in edges['published_at']:
    A_pub[p, v] = 1.0

A_coauth = np.zeros((n_author, n_author))
for a1, a2 in edges['co_author']:
    A_coauth[a1, a2] = 1.0

print('\nA_writes (Author → Paper):')
print(A_writes.astype(int))
print('\nA_pub (Paper → Venue):')
print(A_pub.astype(int))
print('\nA_coauth (Author ↔ Author):')
print(A_coauth.astype(int))


In [ ]:
# (c) RGCN aggregation for paper nodes

d = 2  # feature/hidden dimension
np.random.seed(13)

# Random initial features
H_author = np.random.randn(n_author, d)   # shape (3, 2)
H_paper  = np.random.randn(n_paper, d)    # shape (2, 2)

# Random weight matrices
W_self   = np.random.randn(d, d)   # self-loop weight for paper
W_writes = np.random.randn(d, d)   # relation weight for 'writes'

# RGCN update for paper nodes:
# h_paper = ReLU(W_self @ H_paper + A_writes^T @ H_author @ W_writes)
# Note: shapes — A_writes.T is (2,3), H_author is (3,2), W_writes is (2,2)
agg_from_authors = A_writes.T @ H_author @ W_writes  # (2,3)@(3,2)@(2,2) = (2,2)
self_term = H_paper @ W_self                         # (2,2)@(2,2) = (2,2)
H_paper_new = np.maximum(0, self_term + agg_from_authors)

print(f'H_author  shape: {H_author.shape}')
print(f'H_paper   shape: {H_paper.shape}')
print(f'A_writes  shape: {A_writes.shape}')
print(f'\nRGCN aggregated H_paper^(1) shape: {H_paper_new.shape}')
assert H_paper_new.shape == (n_paper, d), f'Shape error: {H_paper_new.shape}'
print('H_paper^(1) =')
print(H_paper_new.round(4))
print('PASS — RGCN aggregation shape correct')


In [ ]:
# (d) Parameter count comparison

d_in, d_out = 2, 2
n_relations = 3
n_node_types = 3

# RGCN: one weight matrix per relation + one self-loop per node type + bias
params_rgcn_relation = n_relations * d_in * d_out      # relation weights
params_rgcn_self     = n_node_types * d_in * d_out    # self-loop weights
params_rgcn_bias     = n_node_types * d_out           # bias per type
params_rgcn_total    = params_rgcn_relation + params_rgcn_self + params_rgcn_bias

# Homogeneous GCN: one shared weight + bias
params_gcn_W    = d_in * d_out
params_gcn_bias = d_out
params_gcn_total = params_gcn_W + params_gcn_bias

overhead = params_rgcn_total / params_gcn_total

print('Parameter count comparison:')
print(f'  RGCN  — relation weights: {params_rgcn_relation}')
print(f'          self-loop weights: {params_rgcn_self}')
print(f'          biases:            {params_rgcn_bias}')
print(f'          TOTAL:             {params_rgcn_total}')
print(f'\n  GCN   — W + bias:         {params_gcn_total}')
print(f'\n  RGCN overhead: {overhead:.1f}x more parameters than homogeneous GCN')
print()
print('Trade-off: RGCN captures relation-specific semantics but scales poorly with')
print('the number of relations (R * d^2 parameters). For large R, use compositional')
print('encodings (e.g. CompGCN) or basis decomposition to reduce parameter count.')


---

## Summary

| # | Topic | Key Result |
|---|-------|------------|
| 1 ★ | Adjacency Matrix | $A^2$ counts length-2 walks; density = nnz/n² |
| 2 ★ | Adj. List & Edge List | COO = (row, col, data) with 2|E| entries for undirected |
| 3 ★ | Laplacian | $L=D-A$ is PSD; $\lambda_2>0$ ↔ connected; quadratic form = edge sum |
| 4 ★★ | CSR Format | indptr has length n+1; SpMV via row slices; CSR saves memory when dense < 2/3 |
| 5 ★★ | Incidence Matrix | $L = BB^\top$; $B^\top \mathbf{1} = \mathbf{0}$; columns encode signed edge directions |
| 6 ★★ | Normalised Adj. / GCN | $\hat{A}=D^{-1/2}AD^{-1/2}$; spectrum in $[-1,1]$; GCN adds self-loops |
| 7 ★★★ | Memory Benchmark | CSR $\approx 12 \cdot$nnz bytes; crossover density $d^*=2/3$ for float64 |
| 8 ★★★ | Hetero Graphs / RGCN | Per-relation weight matrices; $R \cdot d^2$ param overhead vs shared GCN |

**Next:** §03 Graph Traversal covers BFS, DFS, and shortest-path algorithms using these representations.
